In [1]:
import torch

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-27 14:15:11--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-04-27 14:15:11 (143 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
text = open('input.txt', 'r', encoding='utf-8').read()
vocab = sorted(list(set(text)))
stoi = {c:i for i,c in enumerate(vocab)}
itos = {i:c for i,c in enumerate(vocab)}

encode = lambda input: [stoi[c] for c in input]
decode = lambda input: ''.join([itos[c] for c in input])

data = torch.tensor(encode(text), dtype=torch.long)

In [4]:
# training data - prepare batches

torch.manual_seed(1337)
batch_size = 4 # 4 inputs in parallel
block_size = 8 # max context length
emb_size = 32
vocab_size = len(vocab)

ix = torch.randint(0, len(text)-block_size, (batch_size, 1))
x = torch.stack([data[i:i+block_size] for i in ix])
y = torch.stack([data[i+1:i+block_size+1] for i in ix])


In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()

    self.token_embedding_table = nn.Embedding(vocab_size, emb_size)
    self.positon_embedding_table = nn.Embedding(block_size, emb_size)
    self.lm_head = nn.Linear(emb_size, vocab_size)


  def forward(self, idx, targets=None):

    B, T = idx.shape
    input_embeddings = self.token_embedding_table(idx) # (B, T, C)
    pos_embeddings = self.positon_embedding_table(torch.arange(T)) # (T, C) C = emb_size
    x = input_embeddings + pos_embeddings
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  # Batched generation
  def generate(self, idx, max_new_tokens):
    # idx = (B,T)
    for _ in range(max_new_tokens):

      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size]

      # get the predictions
      logits, loss = self(idx) # logits = (B, T, C)

      # focus only on last character : This is bigram
      logits = logits[:,-1,:] # (B,C) logits is eq to count

      # find prob
      probs = F.softmax(logits, dim=-1) # (B,C)

      # sample
      idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx



m = BigramLanguageModel(vocab_size)
out, loss = m(x, y)
print(out.shape)
print(loss)


torch.Size([32, 65])
tensor(4.6208, grad_fn=<NllLossBackward0>)


In [6]:
# Generate text using above untrained model
# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(m.generate(idx,7)[0].tolist()))


In [7]:
# training
batch_size = 32

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# data loading
def get_batch():
    # generate a small batch of data of inputs x and targets y
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [8]:
# for _ in range(10000):
#   # forward pass - calculate loss
#   x,y = get_batch()
#   logits, loss = m.forward(x,y)

#   # setting grad to zero
#   optimizer.zero_grad(set_to_none=True)

#   # backward pass
#   loss.backward()

#   # updating parameters to reduce the loss
#   optimizer.step()

# print(loss.item())


In [9]:
# Text generation - an example
# idx = torch.zeros((1,1), dtype=torch.long)
# print(decode(m.generate(idx,6)[0].tolist()))


# ==========================
#     **SELF-ATTENTION**
# ==========================


In [10]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [11]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T)) # we don't want all to be zero. Current words closeness with words will vary based on data
wei = wei.masked_fill(tril == 0, float('-inf')) ## Decoder block because of this line # future words not not sending signals to prev words
wei = F.softmax(wei, dim=-1)
print(wei)
xbow3 = wei @ x
#print(xbow3.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [12]:
# version 4: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time(block_size), channels
x = torch.randn(B,T,C)

# Single head performing self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # wei = affinities (B, T, 16) @ (B, 16, T) ---> (B,T,T) T square matrix giving us

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)

out = wei @ v
out.shape

torch.Size([4, 8, 16])

In [13]:
# # hyperparameters
# batch_size = 64 # how many independent sequences will we process in parallel?
# block_size = 256 # what is the maximum context length for predictions?
# max_iters = 5000
# eval_interval = 500
# learning_rate = 3e-4
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# eval_iters = 200
# n_embd = 384
# n_head = 6
# n_layer = 6
# dropout = 0.2
# # ------------

n_embd = 32
emb_size = 32 # n_embd and emb_size both are same. Too lazy to rename now
head_size = 16

In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class Head(nn.Module):
  """ One head of self-attention """

  def __init__(self, head_size):
    super().__init__()

    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.values = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x) # (B,T,C)
    q = self.query(x) # (B,T,C)

    # compute attention scores (affinities)
    wei = q @ k.transpose(-2,-1) * C**0.5 # (B,T,C) @ (B,C,T) ---> (B,T,T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
    wei = F.softmax(wei, dim=-1) # (B,T,T)

    #perform the weighted aggregation of the values
    v = self.values(x) # (B,T,C)
    out = wei @ v # (BTT) @ (BTC) ---> B,T,C
    return out

In [32]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.proj(out)
    return out

In [31]:
class FeedForward(nn.Module):

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
    )

  def forward(self, x):
    return self.net(x)

In [33]:
class BigramLanguageModelWithSA(nn.Module):

  def __init__(self):
    super().__init__()

    self.token_embedding_table = nn.Embedding(vocab_size, emb_size)
    self.positon_embedding_table = nn.Embedding(vocab_size, emb_size)
    #self.sa_head = Head(n_embd) # self attention head
    self.sa_heads = MultiHeadAttention(4, n_embd//4) # self attention heads
    self.ffwd = FeedForward(emb_size)
    self.lm_head = nn.Linear(emb_size, vocab_size)

  def forward(self, idx, targets=None):

    B, T = idx.shape

    input_embeddings = self.token_embedding_table(idx) # (B, T, C)
    pos_embeddings = self.positon_embedding_table(torch.arange(T)) # (T, C) C = emb_size
    x = input_embeddings + pos_embeddings
    x = self.sa_heads(x) # apply heads of self-attention
    x = self.ffwd(x)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx = (B,T)
    for _ in range(max_new_tokens):

      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]

      # get the predictions
      logits, loss = self(idx_cond) # logits = (B, T, C)

      # focus only on last character : This is bigram
      logits = logits[:,-1,:] # (B,C) logits is eq to count

      # find prob
      probs = F.softmax(logits, dim=-1) # (B,C)

      # sample
      idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx



In [18]:
m = BigramLanguageModelWithSA()
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
for _ in range(10000):
  # forward pass - calculate loss
  x,y = get_batch()
  logits, loss = m.forward(x,y)

  # setting grad to zero
  optimizer.zero_grad(set_to_none=True)

  # backward pass
  loss.backward()

  # updating parameters to reduce the loss
  optimizer.step()

print(loss.item())

2.1993632316589355


In [19]:
# Text generation - an example
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx,360)[0].tolist()))


Whath derr de nay cher, fede,
Thaves halt, and hode mor awat cioniay pave for shangem, yous
Tar her is casoul befe my dos be;
I dend wiolgeingger!
Yish
KO
Tar sis lour thind,
Thoveyt, ou haver; thingdacur
DUmalave there'n omanld.

Se ingdest fore me my shespeardide nowand to dase thacking thonand no Lor wone and debur
ye paby akom dele by:
And his lrew thano


In [34]:
class Block(nn.Module):

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd=n_embd)

  def forward(self, x):
    x = x + self.sa(x)
    x = x + self.ffwd(x)
    return x

In [35]:
class BigramLanguageModelWithBlocks(nn.Module):

  def __init__(self):
    super().__init__()

    self.token_embedding_table = nn.Embedding(vocab_size, emb_size)
    self.positon_embedding_table = nn.Embedding(vocab_size, emb_size)
    self.blocks = nn.Sequential(
        Block(n_embd, n_head=4),
        Block(n_embd, n_head=4),
        Block(n_embd, n_head=4)
    )
    self.lm_head = nn.Linear(emb_size, vocab_size)

  def forward(self, idx, targets=None):

    B, T = idx.shape

    input_embeddings = self.token_embedding_table(idx) # (B, T, C)
    pos_embeddings = self.positon_embedding_table(torch.arange(T)) # (T, C) C = emb_size
    x = input_embeddings + pos_embeddings
    x = self.blocks(x) # (B,T,C)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx = (B,T)
    for _ in range(max_new_tokens):

      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]

      # get the predictions
      logits, loss = self(idx_cond) # logits = (B, T, C)

      # focus only on last character : This is bigram
      logits = logits[:,-1,:] # (B,C) logits is eq to count

      # find prob
      probs = F.softmax(logits, dim=-1) # (B,C)

      # sample
      idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx



In [36]:
m = BigramLanguageModelWithBlocks()
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
for _ in range(10000):
  # forward pass - calculate loss
  x,y = get_batch()
  logits, loss = m.forward(x,y)

  # setting grad to zero
  optimizer.zero_grad(set_to_none=True)

  # backward pass
  loss.backward()

  # updating parameters to reduce the loss
  optimizer.step()

print(loss.item())

1.8790842294692993
